In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Check for missing values in the dataset
try:
    dataset
except NameError:
    import pandas as pd
    import os
    if not os.path.exists('insurance.csv'):
        !wget -q https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
    dataset = pd.read_csv('insurance.csv')

missing_values = dataset.isna().sum()
print("Missing values per column:")
print(missing_values)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Convert categorical columns to numeric using dummy encoding for correlation calculation
df_encoded = pd.get_dummies(dataset, columns=['sex', 'smoker', 'region'], drop_first=True)

# Calculate correlation matrix
corr_matrix = df_encoded.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow import keras
import numpy as np

# 1. Prepare data by converting categorical columns to numeric
df_model = dataset.copy()
for col in ['sex', 'smoker', 'region']:
    df_model[col] = df_model[col].astype('category').cat.codes

# 2. Split dataset into train and test sets (80% train, 20% test)
train_dataset = df_model.sample(frac=0.8, random_state=42)
test_dataset = df_model.drop(train_dataset.index)

# 3. Separate target values ('expenses') from features
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

# 4. Normalize the features
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(np.array(train_dataset))

# 5. Build and compile the sequential Keras model
model = keras.Sequential([
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer=tf.optimizers.Adam(learning_rate=0.1),
    loss='mean_absolute_error',
    metrics=['mae', 'mse']
)

# 6. Train the model to achieve an MAE below 3500
history = model.fit(
    train_dataset,
    train_labels,
    epochs=120,
    verbose=1,
    validation_split=0.2
)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
